In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
FILES = {
    "applications": "applications.csv",
    "bureau": "credit_bureau.csv",
    "servicing": "loan_servicing.csv",
    "holdout": "holdout_applications.csv",
}

def load_raw(raw_dir=RAW):
    """Read every raw file into a dict of DataFrames keyed by a short name."""
    return {name: pd.read_csv(raw_dir / fname, low_memory=False)
            for name, fname in FILES.items()}

data = load_raw()
for name, df in data.items():
    print(f"{name:<13}{df.shape}")

applications (30600, 20)
bureau       (30000, 22)
servicing    (30000, 11)
holdout      (5000, 40)


In [2]:
APPLICATION_COLS = [
    "id", "issue_d", "loan_amnt", "term", "int_rate", "installment", "grade",
    "sub_grade", "purpose", "title", "application_type", "emp_title",
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "dti", "zip_code", "addr_state",
]
BUREAU_COLS = [
    "earliest_cr_line", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "delinq_2yrs", "mths_since_last_delinq", "pub_rec", "pub_rec_bankruptcies",
    "open_acc", "total_acc", "revol_bal", "revol_util", "bc_util", "tot_cur_bal",
    "avg_cur_bal", "total_rev_hi_lim", "tot_hi_cred_lim", "mort_acc",
    "acc_open_past_24mths", "num_tl_op_past_12m",
]
SERVICING_COLS = [
    "total_pymnt", "total_rec_prncp", "total_rec_int", "recoveries",
    "last_pymnt_d", "last_pymnt_amnt", "last_credit_pull_d",
    "last_fico_range_high", "last_fico_range_low", "debt_settlement_flag",
]

EXPECTED = {
    "applications": APPLICATION_COLS + ["defaulted"],
    "bureau": ["id"] + BUREAU_COLS,
    "servicing": ["id"] + SERVICING_COLS,
    "holdout": APPLICATION_COLS + BUREAU_COLS,  # no label
}

In [3]:
def compare_columns(df, expected):
    """Columns promised by the brief but absent, and columns present but not promised."""
    actual, expected = set(df.columns), set(expected)
    return {"missing": sorted(expected - actual), "extra": sorted(actual - expected)}

for name, df in data.items():
    print(name, compare_columns(df, EXPECTED[name]))

applications {'missing': [], 'extra': []}
bureau {'missing': [], 'extra': ['bureau_recovery_flag']}
servicing {'missing': [], 'extra': []}
holdout {'missing': [], 'extra': ['bureau_recovery_flag']}


In [4]:
for name, df in data.items():
    text_cols = df.select_dtypes("object").columns.tolist()
    print(f"\n{name}: {len(text_cols)} text columns")
    print(text_cols)


applications: 14 text columns
['id', 'issue_d', 'term', 'grade', 'sub_grade', 'purpose', 'title', 'application_type', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'zip_code', 'addr_state']

bureau: 2 text columns
['id', 'earliest_cr_line']

servicing: 4 text columns
['id', 'last_pymnt_d', 'last_credit_pull_d', 'debt_settlement_flag']

holdout: 15 text columns
['id', 'issue_d', 'term', 'grade', 'sub_grade', 'purpose', 'title', 'application_type', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'zip_code', 'addr_state', 'earliest_cr_line']


/tmp/ipykernel_65359/2342261060.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes("object").columns.tolist()
/tmp/ipykernel_65359/2342261060.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details o

In [5]:
def missing_report(df):
    """Percentage of missing values per column, highest first, hiding zeros."""
    pct = df.isna().mean().mul(100).round(1)
    return pct[pct > 0].sort_values(ascending=False)

for name, df in data.items():
    print(f"\n--- {name} ---")
    print(missing_report(df))


--- applications ---
emp_title     6.0
emp_length    5.5
dtype: float64

--- bureau ---
mths_since_last_delinq    45.1
bc_util                    1.1
dtype: float64

--- servicing ---
last_pymnt_d    0.1
dtype: float64

--- holdout ---
mths_since_last_delinq    47.6
emp_title                  6.8
emp_length                 6.7
title                      5.5
bc_util                    1.3
revol_util                 0.1
dtype: float64


In [6]:
CATEGORICAL = ["term", "grade", "sub_grade", "emp_length", "home_ownership",
               "verification_status", "purpose", "application_type", "defaulted"]

apps = data["applications"]
for col in CATEGORICAL:
    print(f"\n--- {col} ({apps[col].nunique(dropna=False)} distinct) ---")
    print(apps[col].value_counts(dropna=False).head(15))


--- term (2 distinct) ---
term
36 months    22772
60 months     7828
Name: count, dtype: int64

--- grade (7 distinct) ---
grade
B    8805
C    8558
A    5230
D    4756
E    2333
F     743
G     175
Name: count, dtype: int64

--- sub_grade (35 distinct) ---
sub_grade
B4    1958
B3    1889
C1    1818
C3    1797
C2    1764
C4    1721
B2    1719
B5    1657
A5    1583
B1    1582
C5    1458
D1    1283
A4    1166
D2     952
D4     905
Name: count, dtype: int64

--- emp_length (12 distinct) ---
emp_length
10+ years    10309
2 years       2712
3 years       2401
< 1 year      2361
1 year        1920
5 years       1819
4 years       1702
NaN           1693
8 years       1547
7 years       1509
6 years       1401
9 years       1226
Name: count, dtype: int64

--- home_ownership (3 distinct) ---
home_ownership
MORTGAGE    15261
RENT        12154
OWN          3185
Name: count, dtype: int64

--- verification_status (3 distinct) ---
verification_status
Source Verified    11950
Verified            99

In [7]:
for name in ["applications", "bureau"]:
    print(f"\n=== {name} ===")
    display(data[name].select_dtypes("number")
            .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)


=== applications ===


,count,mean,std,min,1%,5%,50%,95%,99%,max
loan_amnt,30600.0,15906.452614,17521.885901,1000.00,1800.0000,3600.000,12950.00,34926.25,35000.0000,350000.00
int_rate,30600.0,13.153676,4.424235,5.32,5.3200,6.490,12.99,20.99,24.9900,28.99
installment,30600.0,439.775885,248.534498,23.26,60.2083,123.029,381.70,929.87,1192.6415,1406.45
annual_inc,30600.0,72850.516754,71596.763147,11.50,50.0000,22000.000,62500.00,150000.00,250000.0000,7500000.00
dti,30600.0,18.290196,8.346786,0.00,2.2900,5.230,17.76,32.87,37.6200,100.09
defaulted,30600.0,0.187288,0.390149,0.00,0.0000,0.000,0.00,1.00,1.0000,1.00



=== bureau ===


,count,mean,std,min,1%,5%,50%,95%,99%,max
fico_range_low,30000.0,693.443000,29.862744,660.0,660.00,660.00,685.0,755.00,795.000,845.0
fico_range_high,30000.0,697.403067,29.859587,660.0,664.00,664.00,689.0,759.00,799.000,850.0
inq_last_6mths,30000.0,0.671433,0.955113,0.0,0.00,0.00,0.0,3.00,4.000,6.0
delinq_2yrs,30000.0,0.332833,0.897189,0.0,0.00,0.00,0.0,2.00,4.000,18.0
mths_since_last_delinq,16465.0,30.770544,23.009992,0.0,0.00,0.00,27.0,73.00,80.000,156.0
pub_rec,30000.0,0.214233,0.566346,0.0,0.00,0.00,0.0,1.00,2.000,11.0
pub_rec_bankruptcies,30000.0,0.134533,0.375818,0.0,0.00,0.00,0.0,1.00,1.000,9.0
open_acc,30000.0,11.666267,5.285744,1.0,3.00,5.00,11.0,22.00,28.000,53.0
total_acc,30000.0,25.483267,11.809486,3.0,6.00,9.00,24.0,47.00,60.000,105.0
revol_bal,30000.0,18338.955233,32812.740273,0.0,496.99,2223.95,11892.0,45837.55,127486.140,1561990.0


In [8]:
issue = pd.to_datetime(apps["issue_d"], errors="coerce", format="mixed")
print("Dates that failed to parse:", issue.isna().sum())
print(issue.dt.year.value_counts().sort_index())

Dates that failed to parse: 0
issue_d
2013     5454
2014     9212
2015    15934
Name: count, dtype: int64


In [ ]:
for col in ["emp_length", "home_ownership", "verification_status",
            "purpose", "application_type"]:
    print(f"\n--- {col} ---")
    print(apps[col].value_counts(dropna=False).to_string())


--- verification_status ---
verification_status
Source Verified    11950
Verified            9965
Not Verified        8685

--- purpose ---
purpose
debt_consolidation    18052
credit_card            7464
home_improvement       1773
other                  1478
major_purchase          509
medical                 303
car                     264
small_business          262
moving                  193
vacation                148
house                   119
wedding                  22
renewable_energy         13
